## Load the transformer-sentiment dataset

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_transformer_sentiment_large.csv")
print(f"Loaded {len(df)} rows")
df.columns.tolist()

Loaded 30000 rows


['reviewId',
 'userName',
 'userImage',
 'review_text',
 'rating',
 'thumbs_up',
 'reviewCreatedVersion',
 'review_date',
 'replyContent',
 'repliedAt',
 'appVersion',
 'got_reply',
 'year',
 'month',
 'review_length',
 'sentiment_score',
 'sentiment_label',
 'themes',
 'transformer_label',
 'transformer_score']

## Create a signed transformer sentiment score

In [2]:
def signed_transformer_score(row):
    if row["transformer_label"] == "POSITIVE":
        return row["transformer_score"]
    else:
        return -row["transformer_score"]

df["transformer_sentiment_signed"] = df.apply(signed_transformer_score, axis=1)
df[["review_text", "transformer_label", "transformer_score", "transformer_sentiment_signed"]].head()

,review_text,transformer_label,transformer_score,transformer_sentiment_signed
0,This Revolut App should not be allowed to prov...,NEGATIVE,0.998738,-0.998738
1,"Absolutely fantastic! Easy to use, friendly in...",POSITIVE,0.999852,0.999852
2,I'm new on this app and tell now it's working ...,POSITIVE,0.999440,0.999440
3,We went away best card I have had,POSITIVE,0.997566,0.997566
4,Perfect,POSITIVE,0.999852,0.999852


## Feature engineering 

In [3]:
df["is_one_star"] = (df["rating"] == 1).astype(int)
df["has_account_freeze"] = df["themes"].str.contains("account_freeze").astype(int)
df["has_customer_support"] = df["themes"].str.contains("customer_support").astype(int)
df["has_verification"] = df["themes"].str.contains("verification").astype(int)
df["has_fees"] = df["themes"].str.contains("fees").astype(int)
df["has_app_stability"] = df["themes"].str.contains("app_stability").astype(int)
df["has_transfers"] = df["themes"].str.contains("transfers").astype(int)

features = ["transformer_sentiment_signed", "review_length", "has_account_freeze", "has_customer_support", 
            "has_verification", "has_fees", "has_app_stability", "has_transfers"]
X = df[features].fillna(0)
y = df["is_one_star"]

print(f"1-star reviews: {y.sum()} out of {len(y)} ({y.mean():.1%})")

1-star reviews: 3680 out of 30000 (12.3%)


## Train and evaluate

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_v2 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
model_v2.fit(X_train, y_train)

y_pred = model_v2.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      5264
           1       0.68      0.68      0.68       736

    accuracy                           0.92      6000
   macro avg       0.82      0.82      0.82      6000
weighted avg       0.92      0.92      0.92      6000



## Feature importance

In [6]:
importance_df_v2 = pd.DataFrame({
    "feature": features,
    "importance": model_v2.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df_v2)

                        feature  importance
0  transformer_sentiment_signed    0.682309
1                 review_length    0.250222
3          has_customer_support    0.024158
2            has_account_freeze    0.015368
4              has_verification    0.012224
6             has_app_stability    0.006932
5                      has_fees    0.004685
7                 has_transfers    0.004103


## Predictive Model v2: Retrained with Transformer Sentiment

**Method:** Retrained the Random Forest Classifier from before, replacing the VADER-based 
`sentiment_score` feature with a transformer-derived signed sentiment score 
(`transformer_sentiment_signed` — combining the transformer's label and confidence into 
one comparable positive/negative number, mirroring VADER's original -1 to +1 scale).

### Model Comparison: VADER-based vs. Transformer-based

| Metric (1-star class) | VADER Model | Transformer Model | Change |
|---|---|---|---|
| Precision | 0.54 | 0.68 | +0.14 |
| Recall | 0.69 | 0.68 | -0.01 |
| F1-score | 0.60 | 0.68 | +0.08 |
| Overall accuracy | 0.89 | 0.92 | +0.03 |

**Interpretation:** switching to transformer-based sentiment improved precision substantially 
(fewer false positives when the model flags a review as 1-star) and overall F1-score, while 
recall remained essentially unchanged. This is a genuine, broad improvement rather than a 
trade-off between metrics.

### Feature Importance (Transformer-based model)

| Feature | Importance |
|---|---|
| transformer_sentiment_signed | 68.2% |
| review_length | 25.0% |
| has_customer_support | 2.4% |
| has_account_freeze | 1.5% |
| has_verification | 1.2% |
| has_app_stability | 0.7% |
| has_fees | 0.5% |
| has_transfers | 0.4% |

**customer_support remains the top-ranked specific theme**, consistent with the earlier 
VADER-based model — reinforcing this as a stable, reliable finding rather than an artifact 
of one particular method.

### Triangulated Finding

Customer support has now emerged as the most significant specific complaint driver across 
**three independent analytical methods**:
1. Manual keyword frequency analysis (17.7% of 1-star reviews)
2. Unsupervised BERTopic modeling (Topic 15: account/blocked/closed)
3. Predictive feature importance (top-ranked theme in both model versions)

This convergence across unrelated methods provides strong evidence that customer support 
is a genuine, high-priority driver of negative sentiment — not a coincidence of how any 
single method was designed.

### Methodological Takeaway

This iteration demonstrates a complete analytical improvement cycle: an initial method 
(VADER) was used, benchmarked against ground truth, found to have real limitations, and 
replaced with a more accurate method (transformer sentiment) — resulting in measurable 
improvement to the downstream predictive model built on top of it.